In [0]:
%pip install langchain-anthropic==1.3.4

In [0]:
%pip install --upgrade databricks-langchain

In [0]:
from langchain.tools import tool
import requests
import os

Create a basic Agent

In [0]:
def create_calculator(a: int, b: int, operator: str):
    """
    Perform arithmetic calculation"""
    return eval(f"{a} {operator} {b}")

In [0]:
def get_weather(city: str):
    """Get temperature of city in celcius"""
    url = "http://api.weatherapi.com/v1/current.json"
    WEATHER_API_KEY = dbutils.widgets.get(scope = "scope_development", key = "WEATHER_API_KEY")
    params = {"key":WEATHER_API_KEY, "q":city, "aqi":"no"}
    response = requests.get(url, params = params)
    return response.json()['current']['temp_c']

In [0]:
from langchain.agents import create_agent
import os

os.environ["ANTHROPIC_API_KEY"] = dbutils.secrets.get(scope = "scope_development", key = "ANTHROPIC_API_KEY")

agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[create_calculator, get_weather],
    system_prompt="You are a helpful assistant. Don't add any extra messages from your side. Just call the tools and give me the output",
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather of Puri"}]}
)
print(response["messages"][-1].content)

In [0]:
from databricks_langchain import ChatDatabricks
from langchain.agents import create_agent
from langchain.tools import tool
import requests

# Define system prompt
SYSTEM_PROMPT = """You are an expert weather forecaster.

- get_weather: use this to get the weather for a specific location

If you get irrelevant questions then reply 'I don't have relevant context for this  question'
"""

@tool
def get_weather(city: str):
    """Get temperature of city in celcius"""
    url = "http://api.weatherapi.com/v1/current.json"
    WEATHER_API_KEY = dbutils.secrets.get("scope_development", "WEATHER_API_KEY")
    params = {"key":WEATHER_API_KEY, "q":city, "aqi":"no"}
    response = requests.get(url, params = params)
    return response.json()['current']['temp_c']

model = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0
)

agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="You are a helpful weather assistant."
)

response = agent.invoke({"messages": [{"role": "user", "content": "What's the weather in Bhubaneswar?"}, {"role": "user", "content": "What is my IP address"}]})
print(response["messages"][-1].content)

Create a basic agent with in memory context and Structured Output

In [0]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user information based on user ID."""
    print(f"User ID: {runtime.context.user_id}")
    user_id = runtime.context.user_id
    return "Florida" if user_id == "1" else "SF"

In [0]:
# from langchain.tools import ToolRuntime

# print(get_weather_for_location.invoke({"city":"Bhubaneswar"}))

# # Create runtime context with user_id
# runtime = ToolRuntime(context=Context(user_id="1"), state=None, config=None, stream_writer=None, tool_call_id=None, store=None)
# print(get_user_location.invoke({"runtime": runtime}))

In [0]:
from dataclasses import dataclass

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.structured_output import ToolStrategy


# Define system prompt
SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""

# Define context schema
@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

# Define tools
@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user information based on user ID."""
    user_id = runtime.context.user_id
    return "Bhubaneswar" if user_id == "1" else "SF"

# Configure model
model = init_chat_model(
    "claude-sonnet-4-6",
    temperature=0
)

# Define response format
@dataclass
class ResponseFormat:
    """Response schema for the agent."""
    # A punny response (always required)
    punny_response: str
    # Any interesting information about the weather if available
    weather_conditions: str | None = None

# Set up memory
checkpointer = InMemorySaver()

# Create agent
agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[get_user_location, get_weather_for_location],
    context_schema=Context,
    response_format=ToolStrategy(ResponseFormat),
    checkpointer=checkpointer
)

# Run agent
# `thread_id` is a unique identifier for a given conversation.
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="Florida is still having a 'sun-derful' day! The sunshine is playing 'ray-dio' hits all day long! I'd say it's the perfect weather for some 'solar-bration'! If you were hoping for rain, I'm afraid that idea is all 'washed up' - the forecast remains 'clear-ly' brilliant!",
#     weather_conditions="It's always sunny in Florida!"
# )


# Note that we can continue the conversation using the same `thread_id`.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config,
    context=Context(user_id="1")
)

print(response['structured_response'])
# ResponseFormat(
#     punny_response="You're 'thund-erfully' welcome! It's always a 'breeze' to help you stay 'current' with the weather. I'm just 'cloud'-ing around waiting to 'shower' you with more forecasts whenever you need them. Have a 'sun-sational' day in the Florida sunshine!",
#     weather_conditions=None
# )